In [6]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

ENV_PATH = './beauty-agent/.env'

# .env 파일에서 환경 변수 로드
load_dotenv(ENV_PATH)

True

In [32]:
# init_chat_model 사용 (권장)
llms = [
    init_chat_model("openai:gpt-5-mini", temperature=0),
    init_chat_model("google_genai:gemini-2.5-flash", temperature=0)
    ]

In [27]:
gpt_response = llms[0].invoke([HumanMessage(content="LangGraph의 핵심 개념을 설명해주세요.")])
print(gpt_response.content)

어떤 맥락의 LangGraph를 말씀하시는지 확인하고 싶습니다. 특정 라이브러리나 논문(예: GitHub 프로젝트 링크)이 있다면 알려주시면 그에 맞춰 정확히 설명드릴게요.  
일반적으로 “LangGraph”라는 이름으로 얘기될 수 있는 개념(즉, 언어 모델(LLM)과 그래프 구조(지식 그래프·관계 그래프)를 결합한 시스템)의 핵심 개념을 정리해 드리면 다음과 같습니다.

핵심 개념 요약
- 그래프 구조(데이터 모델)
  - 노드(nodes), 엣지(edges), 라벨/타입, 속성(properties). 엔터티와 관계를 구조화해서 표현.
  - 온톨로지/스키마: 노드·관계 유형과 제약을 정의해 일관성 유지.

- 그래프 구축(수집·정규화)
  - 정보추출(IE): 텍스트에서 엔터티 추출(NER), 관계 추출, 이벤트 추출.
  - 정규화/엔티티링킹(entity linking): 같은 실체를 하나의 노드로 통합(동일성 해결).
  - 증거·출처 연결: 각 노드/엣지에 소스와 신뢰도 메타데이터 보관.

- 임베딩과 벡터화
  - 노드·문서·문장 임베딩으로 의미적 유사성 측정.
  - 그래프 + 벡터 하이브리드 검색: 구조적 쿼리(경로, 패턴) + 의미 검색(유사도).

- 검색 및 맥락 구성(RAG 스타일)
  - LLM에 줄 맥락(context)을 구성할 때, 서브그래프나 관련 노드 집합을 검색해 제공.
  - 서브그래프 선택 전략: 중요도, 연결도, 신뢰도 기반 필터링.

- 질의 및 탐색
  - 그래프 쿼리 언어(Cypher, SPARQL 등) 또는 API로 패턴 매칭·경로 탐색 수행.
  - 그래프 탐색을 LLM의 추론 과정과 결합(예: "다음으로 무엇을 조회할지"를 LLM이 결정).

- 추론 및 논리
  - 규칙 기반 추론(논리규칙, 트리플 추론)과 통계적 추론(GNN, 임베딩 기반) 병행.
  - LLM을 이용한 체계적 추론 보조(체인 오브 생각, 증거 기반 응답 생성).

- 에이전트·도구 통합
  - LLM 에이전트가 그래프 조회·수정(노드 추가/

In [36]:
gemini_response = llms[1].invoke([HumanMessage(content="LangGraph의 핵심 개념을 설명해주세요.")])
print(gemini_response.content)

LangGraph는 LangChain을 기반으로 하는 라이브러리로, 복잡한 LLM 애플리케이션, 특히 '에이전트'를 구축하기 위한 강력한 프레임워크입니다. 핵심은 **유한 상태 머신(Finite State Machine)**과 **그래프 이론**을 활용하여 다단계 추론, 도구 사용, 그리고 반복적인 의사결정 과정을 명확하고 제어 가능하게 만드는 것입니다.

기존 LangChain 체인(Chains)은 선형적인 흐름에 강하지만, 에이전트처럼 동적으로 다음 단계를 결정하고, 이전 상태를 기억하며, 특정 조건에 따라 반복적인 작업을 수행하는 데는 한계가 있습니다. LangGraph는 이러한 복잡한 '에이전트적' 행동을 구조화하고 시각화하며 제어할 수 있도록 돕습니다.

LangGraph의 핵심 개념은 다음과 같습니다.

---

### LangGraph의 핵심 개념

1.  **State (상태)**
    *   **가장 근본적인 개념입니다.** LangGraph는 그래프의 각 노드를 통과하면서 업데이트되고 유지되는 **공유 데이터 객체**를 중심으로 작동합니다.
    *   이 상태는 LLM의 응답, 사용된 도구의 결과, 중간 추론 단계, 대화 기록 등 모든 관련 정보를 포함할 수 있습니다.
    *   이를 통해 에이전트는 이전 단계를 '기억'하고, 현재 상태를 기반으로 다음 행동을 결정할 수 있습니다.
    *   일반적으로 `TypedDict`를 상속받아 정의하며, 각 키는 특정 유형의 데이터를 나타냅니다.

2.  **Nodes (노드)**
    *   그래프 내에서 특정 작업을 수행하는 개별 단위입니다.
    *   각 노드는 함수나 Runnable 객체(LangChain의 구성 요소)가 될 수 있습니다.
    *   **예시:**
        *   **LLM 호출 노드:** 사용자 질문에 답변하거나 다음 행동을 계획합니다.
        *   **Tool 호출 노드:** 웹 검색, 데이터베이스 조회, API 호출 등 외부 도구를 사용합